# PyTorch Relay 推理

In [ ]:
import sys

sys.path.extend(["../../../../tests"])
# from tools.tag_span import _create_span, _set_span, _verify_structural_equal_with_span
from tools.torch_utils import verify_model

In [ ]:
import logging
import numpy as np
import torch
import tvm
logging.basicConfig(level=logging.DEBUG)

In [ ]:
def list_ops(expr):
    """list_ops"""

    class OpLister(tvm.relay.ExprVisitor):
        """OpLister inherits from ExprVisitor"""

        def visit_op(self, op):
            if op not in self.node_set:
                self.node_list.append(op)
            return super().visit_op(op)

        def list_nodes(self, expr):
            self.node_set = {}
            self.node_list = []
            self.visit(expr)
            return self.node_list

    return OpLister().list_nodes(expr)

## `torch.matmul`

In [ ]:
class MatMul1(torch.nn.Module):
    def forward(self, *args):
        return torch.matmul(args[0], args[1])

In [ ]:
# vector x vector - 1D x 1D
tensor1 = torch.randn(4)
tensor2 = torch.randn(4)
verify_model(MatMul1().float().eval(), input_data=[tensor1, tensor2], expected_ops=["nn.dense"])